# Visualize Mention Representation

This notebook samples and visualizes a random mention representation from the `mention_representations.parquet` file.

The representation includes:
- **Document Context**: 8 sentences from the document (excluding mention's sentence)
- **Mention Context**: The mention's sentence with `[E]` and `[/E]` markers
- **String Representation**: Combined format ready for tokenization

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set random seed for reproducibility (comment out for different sample each time)
np.random.seed(71)

## Load Data

In [2]:
# Load mention representations
data_path = Path("../data/03_primary/mention_representations.parquet")
df = pd.read_parquet(data_path)

print(f"Total mentions: {len(df):,}")
print(f"Columns: {list(df.columns)}")

Total mentions: 116,793
Columns: ['mention_id', 'string_representation', 'document_context', 'mention_context']


## Sample Random Mention

In [3]:
# Sample one random row
random_row = df.sample(n=1).iloc[0]

print("=" * 80)
print("RANDOM MENTION SAMPLE")
print("=" * 80)

RANDOM MENTION SAMPLE


## Mention ID

In [4]:
print(f"\nMention ID: {random_row['mention_id']}")


Mention ID: R2w83Z6XbYE95GkF3JGGAz


## Document Context (8 sentences)

In [6]:
print("\n" + "=" * 80)
print("DOCUMENT CONTEXT (8 sentences, excluding mention's sentence)")
print("=" * 80)
print(f"\n{random_row['document_context']}")

# Show character count
doc_len = len(random_row['document_context'])
print(f"\n[Length: {doc_len:,} characters]")


DOCUMENT CONTEXT (8 sentences, excluding mention's sentence)

KENYA : Zairean jets bomb rebel - held city , two wounded . Zairean jets bomb rebel - held city , two wounded . Corinne Dufka NAIROBI 1997 - 02 - 17 Three Zairean jet fighters bombed the rebel - held east Zairean city of Bukavu on Monday but caused little damage , aid workers and residents said . The bombing was the first government aerial activity confirmed by independent sources since rebels launched a military campaign last October and seized a large swath of territory in the vast central African nation . An aid worker speaking to Reuters by telephone from Bukavu said loud explosions echoed through Bukavu and panic - stricken residents were seen running for cover . " Bukavu has just been bombed by two jet fighters .

[Length: 728 characters]


## Mention Context (with [E] and [/E] markers)

In [8]:
print("\n" + "=" * 80)
print("MENTION CONTEXT (sentence with [E] mention [/E] markers)")
print("=" * 80)
print(f"\n{random_row['mention_context']}")

# Show character count
mention_len = len(random_row['mention_context'])
print(f"\n[Length: {mention_len:,} characters]")

# Highlight the mention
import re
mention_match = re.search(r'\[E\](.*?)\[/E\]', random_row['mention_context'])
if mention_match:
    mention_text = mention_match.group(1).strip()
    print(f"\n✓ Extracted mention: '{mention_text}'")


MENTION CONTEXT (sentence with [E] mention [/E] markers)

The military says it is reinforcing Zaires third city of Kisangani and [E] Buta [/E] to the north and Kindu to the south .

[Length: 122 characters]

✓ Extracted mention: 'Buta'


## Full String Representation (for tokenizer)

In [10]:
print("\n" + "=" * 80)
print("FULL STRING REPRESENTATION (document_context [SEP] mention_context)")
print("=" * 80)
print(f"\n{random_row['string_representation']}")

# Show total character count
total_len = len(random_row['string_representation'])
print(f"\n[Total length: {total_len:,} characters]")


FULL STRING REPRESENTATION (document_context [SEP] mention_context)

KENYA : Zairean jets bomb rebel - held city , two wounded . Zairean jets bomb rebel - held city , two wounded . Corinne Dufka NAIROBI 1997 - 02 - 17 Three Zairean jet fighters bombed the rebel - held east Zairean city of Bukavu on Monday but caused little damage , aid workers and residents said . The bombing was the first government aerial activity confirmed by independent sources since rebels launched a military campaign last October and seized a large swath of territory in the vast central African nation . An aid worker speaking to Reuters by telephone from Bukavu said loud explosions echoed through Bukavu and panic - stricken residents were seen running for cover . " Bukavu has just been bombed by two jet fighters . [SEP] The military says it is reinforcing Zaires third city of Kisangani and [E] Buta [/E] to the north and Kindu to the south .

[Total length: 857 characters]


## Tokenization Preview (estimate)

Approximate token count (1 token ≈ 4 characters for English text)

In [11]:
# Rough token estimate
estimated_tokens = total_len // 4

print("\n" + "=" * 80)
print("TOKENIZATION PREVIEW")
print("=" * 80)
print(f"\nEstimated tokens: ~{estimated_tokens}")
print(f"Token limit: 128")

if estimated_tokens > 128:
    print(f"\n⚠️  Will require truncation (estimated {estimated_tokens - 128} tokens over limit)")
    print("   → Document context will be truncated first (truncation='only_first')")
    print("   → Mention context with [E] and [/E] markers will be preserved")
else:
    print(f"\n✓ Fits within 128-token limit ({128 - estimated_tokens} tokens remaining)")


TOKENIZATION PREVIEW

Estimated tokens: ~214
Token limit: 128

⚠️  Will require truncation (estimated 86 tokens over limit)
   → Document context will be truncated first (truncation='only_first')
   → Mention context with [E] and [/E] markers will be preserved


## Statistics

In [ ]:
print("\n" + "=" * 80)
print("DATASET STATISTICS")
print("=" * 80)

# Overall stats
print(f"\nTotal mentions: {len(df):,}")
print(f"\nString representation lengths:")
print(df['string_representation'].str.len().describe())

# Check markers
has_start = df['mention_context'].str.contains('[E]', regex=False).sum()
has_end = df['mention_context'].str.contains('[/E]', regex=False).sum()
print(f"\nMarker verification:")
print(f"  [E] markers: {has_start:,} ({has_start/len(df)*100:.1f}%)")
print(f"  [/E] markers: {has_end:,} ({has_end/len(df)*100:.1f}%)")

## Sample Another Random Mention

Run the cells below to see a different random sample

In [ ]:
# Sample another random row (no seed)
another_row = df.sample(n=1).iloc[0]

print("=" * 80)
print("ANOTHER RANDOM SAMPLE")
print("=" * 80)
print(f"\nMention ID: {another_row['mention_id']}")
print(f"\nMention context:")
print(another_row['mention_context'])

# Extract mention
mention_match = re.search(r'\[E\](.*?)\[/E\]', another_row['mention_context'])
if mention_match:
    print(f"\n✓ Mention: '{mention_match.group(1).strip()}'")

print(f"\nTotal length: {len(another_row['string_representation']):,} characters")
print(f"Estimated tokens: ~{len(another_row['string_representation']) // 4}")

## View Multiple Samples

In [ ]:
# Sample 5 random mentions
n_samples = 5
samples = df.sample(n=n_samples)

print("=" * 80)
print(f"SAMPLE OF {n_samples} RANDOM MENTIONS")
print("=" * 80)

for idx, (_, row) in enumerate(samples.iterrows(), 1):
    print(f"\n[{idx}] {row['mention_id']}")
    print(f"    Mention context: {row['mention_context'][:100]}...")
    
    # Extract mention
    mention_match = re.search(r'\[E\](.*?)\[/E\]', row['mention_context'])
    if mention_match:
        print(f"    → Mention: '{mention_match.group(1).strip()}'")
    
    print(f"    → Length: {len(row['string_representation']):,} chars (~{len(row['string_representation'])//4} tokens)")

## Search for Specific Mention by ID

In [ ]:
# Enter a mention_id to look up
mention_id_to_find = "abcnews.go.com_454447"  # Change this to any mention_id

if mention_id_to_find in df['mention_id'].values:
    specific_row = df[df['mention_id'] == mention_id_to_find].iloc[0]
    
    print("=" * 80)
    print(f"SPECIFIC MENTION: {mention_id_to_find}")
    print("=" * 80)
    
    print(f"\nDocument context:")
    print(specific_row['document_context'])
    
    print(f"\n\nMention context:")
    print(specific_row['mention_context'])
    
    # Extract mention
    mention_match = re.search(r'\[E\](.*?)\[/E\]', specific_row['mention_context'])
    if mention_match:
        print(f"\n✓ Mention: '{mention_match.group(1).strip()}'")
else:
    print(f"❌ Mention ID '{mention_id_to_find}' not found in dataset")
    print(f"\nTry one of these IDs:")
    for mid in df['mention_id'].head(10).values:
        print(f"  - {mid}")